In [6]:
import cv2
import json

def precision_artifact_selector(video_path, output_json):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    current_frame = 0
    all_data = []

    print("--- Precision Controls ---")
    print("D: Next Frame | A: Previous Frame")
    print("W: Forward 30 Frames | S: Back 30 Frames")
    print("R: Select Artifacts on current frame")
    print("Q: Save and Exit")
    print("K: Terminate Process")

    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame)
        ret, frame = cap.read()
        if not ret: break

        # Display info on the frame
        display_frame = frame.copy()
        cv2.putText(display_frame, f"Frame: {current_frame}/{total_frames} ({round(current_frame/fps, 2)}s)", 
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        
        cv2.imshow("Precision Selector", display_frame)
        
        key = cv2.waitKey(0) & 0xFF

        if key == ord('d'): # Next
            current_frame = min(current_frame + 1, total_frames - 1)
        elif key == ord('a'): # Back
            current_frame = max(current_frame - 1, 0)
        elif key == ord('w'): # Jump forward
            current_frame = min(current_frame + 30, total_frames - 1)
        elif key == ord('s'): # Jump back
            current_frame = max(current_frame - 30, 0)
        
        elif key == ord('r'): # Record Artifacts
            # This opens a new window specifically for drawing
            roi_list = cv2.selectROIs("Select Multiple (Enter after each, Esc when done)", frame)
            
            if len(roi_list) > 0:
                artifacts = []
                for r in roi_list:
                    artifacts.append({"x": int(r[0]), "y": int(r[1]), "w": int(r[2]), "h": int(r[3])})
                
                all_data.append({
                    "frame": current_frame,
                    "timestamp": round(current_frame / fps, 3),
                    "artifacts": artifacts
                })
                print(f"Saved {len(artifacts)} artifacts for frame {current_frame}")

        elif key == ord('q'):
            break
        elif key == ord('k'):
            cap.release()
            cv2.destroyAllWindows()
            return

    cap.release()
    cv2.destroyAllWindows()

    with open(output_json, 'w') as f:
        json.dump(all_data, f, indent=4)
    print(f"Data exported to {output_json}")

precision_artifact_selector("DataSets/GenVideo-Val/GenVideo-Val/Fake/MoonValley/MoonValley_80.mp4", "precision_log.json")

--- Precision Controls ---
D: Next Frame | A: Previous Frame
W: Forward 30 Frames | S: Back 30 Frames
R: Select Artifacts on current frame
Q: Save and Exit
K: Terminate Process
Saved 2 artifacts for frame 10


In [7]:
import cv2
import json
def display_artifact_overlay(video_path, json_path):
    # 1. Load the recorded data
    try:
        with open(json_path, 'r') as f:
            artifact_log = json.load(f)
    except FileNotFoundError:
        print(f"Error: {json_path} not found.")
        return

    # Convert list to a dictionary for faster lookup: {frame_id: [artifacts]}
    data_lookup = {entry['frame']: entry['artifacts'] for entry in artifact_log}

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    print("Playing video with overlays... Press 'Q' to stop.")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        current_frame = int(cap.get(cv2.CAP_PROP_POS_FRAMES))

        # 2. Check if this frame has any artifacts recorded
        if current_frame in data_lookup:
            for art in data_lookup[current_frame]:
                x, y, w, h = art['x'], art['y'], art['w'], art['h']
                
                # Draw the rectangle
                # (frame, top-left, bottom-right, color, thickness)
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 3)
                
                # Add a label
                cv2.putText(frame, "ARTIFACT", (x, y - 10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        # Display the result
        cv2.imshow("Artifact Overlay Review", frame)
        cv2.moveWindow("Artifact Overlay Review", 100, 100)
        # Press 'q' to exit the review early
        if cv2.waitKey(int(10000/fps)) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# --- To use it ---
display_artifact_overlay("DataSets/GenVideo-Val/GenVideo-Val/Fake/MoonValley/MoonValley_80.mp4", "precision_log.json")

Playing video with overlays... Press 'Q' to stop.
